# 01 - MCP

In this notebook we'll be exploring the Model Context Protocol (MCP). 
* First we'll cover some background on MCP - what is MCP and why it might be useful to us.
* Then we'll build our very own MCP server and connect it to our agent
* Finally, we'll learn how to connect our agent to the wide library of publicly available MCP servers


## What is Model Context Protocol (MCP)
Anthropic (the makers of MCP) defined MCP as "An open protocol that standardises how your LLM applications connect to and work with your tools and data sources"

Think of MCP as a USB connector that we are very familiar with. It allows you to iterface, say your laptop, with several other perepherals without really worrying about how the connect works - it just works! Without the USB standard, each perepheral would come with it's own adapter, cable etc. A nightmare! In fact, Apple was rather infamously doing just this until the EU forced it to standarize their iPhone chargers (which now default to USB 3 protocols!), which means you can use ant standard charger with your latest iPhone.

Consider another example of how the JDBC/ODBC protocols decoupled applications from databases. Before JDBC/ODBC, if you wanted to switch your application from an Oracle database to an MS SQL database, you often had to rewrite the entire data-access layer of your code because the libraries, function calls, and error handling were completely different. With JDBC/ODBC, you may have to just change the way you connect to the database - rest of the interface API remains the same. Similarly with agents before the advent of MCP - if you wanted to give an AI agent access to "Google Drive," you had to write custom integration code specifically for that agent. If you then moved the "drive" to Microsoft 365, you had to write it all over again. MCP turns "tools" into plug-and-play servers.

Also, consider a custom tool (function) you built for your Agent. Let's say it handles a payment gateway interface enabling your Agent to send & receive payments. Say another developer in your organization (or another organization) wants to enable similar functionality, the she would have to write a similar tool. What if you wrote a MCP server as a standard Payment Gateway? Now suddenly all developers in your organization use the same server to enable their agents with Payments functionality - no code duplication. If your MCP server then goes "public" all developers everywhere can use the same functionality without writing custom tools!

In all the examples above, your application is a _host_ that uses a _client_ (interfacing code - JDBC/ODBC) to interface with the _server_ (Database). 


Now let's drill down to some specifics:
* An MCP host _hosts_ an MCP client, which commun

## Using Local MCP Server

In [1]:
from dotenv import load_dotenv

load_dotenv(override=True)

True

> **NOTE**:
>
> There is a **known Windows + Jupyter notebook incompatibility issue** when MCP tries to launch subprocess-based servers, like we will do in this notebook.<br/><br/>


If you **do not** implement the fix in the next cell and run the cells thereafter you'll encounter two cascading errors:
1. **Error 1:** `NotImplementedError` (async subprocess)
MCP first tries `anyio.open_process()` → `asyncio.create_subprocess_exec()`.

On Windows, async subprocess creation requires `ProactorEventLoop`. Jupyter notebooks run their own event loop (typically a `SelectorEventLoop` or one managed by `nest_asyncio`), which doesn't support subprocesses. Therefore base `class BaseEventLoop._make_subprocess_transport` explicitly raises `NotImplementedError` for this case.

2. **Error 2:** `UnsupportedOperation: fileno` (sync fallback)
MCP catches the first error and falls back to plain `subprocess.Popen`. This fails because:

* Jupyter redirects and wraps `stdout/stderr` with custom stream objects (to capture cell output)
* These wrapper streams don't have a real OS file descriptor, so `fileno()` raises `UnsupportedOperation`
`subprocess.Popen` needs a real file descriptor when you pass `stderr=errlog`

To fix these errors, we must run the code in the following cell **_before_** calling any MCP server code. It has a `sys.platform == "win32"` _guard_, which means it will run only if your OS is Windows.

In [2]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(
        asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy
    ):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
            "transport": "stdio",
            "command": "python",
            "args": ["resources/01_mcp_server.py"],
        },
    }
)

In [5]:
# get the tools, resources & prompts that our local server provides
tools = await client.get_tools()
print("-------- list of tools available -------- ")
print(tools)

# get resources
resources = await client.get_resources("local_server")
print("-------- resources available -------- ")
print(resources)

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content
print("-------- prompt available -------- ")
print(prompt)

-------- list of tools available -------- 
[StructuredTool(name='web_search', description='Search the web for information based on query', args_schema={'properties': {'query': {'title': 'Query', 'type': 'string'}}, 'required': ['query'], 'title': 'web_searchArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x0000021CE156F2E0>)]
-------- resources available -------- 
[Blob 2323065712352]
-------- prompt available -------- 

    You are a helpful assistant that answers questions about LangChain, LangGraph and LangSmith.
    
    You can use the following tools/resources to answe user's questions:
    - search_web: search the web for information.
    - github_file: Access the langchain-ai repo files.

    If user user asks a question that is NOT RELATED to LangChain, LangGraph or LangSmith, you
    should respond with "I am sorry, I can only answer questions related to LangChain, LangGraph an

In [6]:
# create our agent the usual way
from langchain.agents import create_agent

agent = create_agent(
    model="openai:gpt-5-nano",
    # tools from MCP server - is a list!
    tools=tools,
    # system prompt also from MCP server
    system_prompt=prompt,
)

Now let's ask it some questions about LangChain/LangGraph/LangSmith

In [ ]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

# NOTE: we use aiinvoke() instead of invoke()
response = await agent.ainvoke(
    {
        "messages": [
            HumanMessage(content="Tell me about the langchain-mcp-adapters library")
        ]
    },
    config=config,
)
print(" ----- Response from MCP server ----- ")
# to view all the messages exchanges use the following 2 lines
# from pprint import pprint
# pprint(response)
print(response["messages"][-1].content)

 ----- Response from MCP server ----- 
Here’s a concise overview of the LangChain MCP Adapters library.

What it is
- A bridge between Anthropic’s Model Context Protocol (MCP) and LangChain/LangGraph.
- It lets you connect to MCP tool servers and present their tools, prompts, and resources as LangChain-compatible elements.
- It supports reading from multiple MCP servers at once and loading their tools into LangChain workflows (agents, tools, prompts, and resources).
- It includes adapters for converting MCP content into LangChain/Blob content, prompts into LangChain messages, and tools into LangChain tools.
- The project is designed to work with LangChain, LangGraph, and LangSmith for tooling, orchestration, and logging.

Key components and features
- Tools adapter: converts MCP tools into LangChain-compatible tools so you can plug MCP tools into LangChain agents.
- Prompts adapter: converts MCP prompts to LangChain messages.
- Resources adapter: converts MCP resources into LangChain B

Cool! It was able to get me a response from the local MCP server.

## Online MCP
Next we'll attempt to use an online MCP time server that tells you the time in New York.

In [9]:
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uvx",
            "args": ["mcp-server-time", "--local-timezone=America/New_York"],
        }
    }
)

online_tools = await client.get_tools()
print("-------- list of tools available -------- ")
print(tools)

-------- list of tools available -------- 
[StructuredTool(name='get_current_time', description='Get current time in a specific timezones', args_schema={'type': 'object', 'properties': {'timezone': {'type': 'string', 'description': "IANA timezone name (e.g., 'America/New_York', 'Europe/London'). Use 'America/New_York' as local timezone if no timezone provided by the user."}}, 'required': ['timezone']}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x0000021CF9F94CC0>), StructuredTool(name='convert_time', description='Convert time between timezones', args_schema={'type': 'object', 'properties': {'source_timezone': {'type': 'string', 'description': "Source IANA timezone name (e.g., 'America/New_York', 'Europe/London'). Use 'America/New_York' as local timezone if no source timezone provided by the user."}, 'time': {'type': 'string', 'description': 'Time to convert in 24-hour format (HH:MM)'}, 'target_timezone': {'type'

In [ ]:
time_agent = create_agent(
    model="openai:gpt-5-nano",
    tools=online_tools,
    system_prompt="""You can fetch time from New York using tools available 
    to you. Display just the time & nothing else. Use the following format:

    It’s 12:36 PM on Friday, June 28, 2019 in New York (Eastern Daylight Time, UTC-4)
    """,
)

In [14]:
# let's ask for time in New York
question = HumanMessage(content="What time is it?")

response = await time_agent.ainvoke({"messages": [question]})

print(" ----- Response from MCP Time server ----- ")
# to view all the messages exchanges use the following 2 lines
# from pprint import pprint
# pprint(response)
print(response["messages"][-1].content)

 ----- Response from MCP Time server ----- 
It’s 2:38 AM on Friday, April 17, 2026 in New York (Eastern Daylight Time, UTC-4)
